# Module 03 — Storage I/O Performance

Compare read/write performance across formats.

In [ ]:
import sys
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import get_spark
from shared import config
from shared.utils import timed_action

spark = get_spark('module-03-storage-io')
trips = spark.read.parquet(*config.yellow_taxi_paths())
sample = trips.limit(500_000)

## Read Benchmarks

In [ ]:
out = config.OUTPUT_BASE / '03-read-bench'
out.mkdir(parents=True, exist_ok=True)

# Write sample in multiple formats for fair read comparison
sample.write.mode('overwrite').json(str(out / 'json'))
sample.write.mode('overwrite').parquet(str(out / 'parquet'))
sample.writeTo('local.default.taxi_sample').using('iceberg').createOrReplace()

In [ ]:
results = {}

# JSON read
_, t = timed_action(spark.read.json(str(out / 'json')))
results['json'] = t

# Parquet serial
spark.conf.set('spark.sql.parquet.enableVectorizedReader', 'false')
_, t = timed_action(spark.read.parquet(str(out / 'parquet')))
results['parquet_serial'] = t

# Parquet vectorized
spark.conf.set('spark.sql.parquet.enableVectorizedReader', 'true')
_, t = timed_action(spark.read.parquet(str(out / 'parquet')))
results['parquet_vectorized'] = t

# Iceberg
_, t = timed_action(spark.table('local.default.taxi_sample'))
results['iceberg'] = t

for fmt, sec in sorted(results.items(), key=lambda x: x[1]):
    print(f'{fmt:20s} {sec:.2f}s')

## Iceberg Filter Pushdown

Compare files scanned with and without a filter.

In [ ]:
spark.table('local.default.taxi_sample').filter('total_amount > 100').explain('formatted')
# Iceberg skips files using column stats in metadata

## Write Benchmarks

In [ ]:
write_out = config.OUTPUT_BASE / '03-write-bench'
write_out.mkdir(parents=True, exist_ok=True)

write_results = {}

for fmt, writer in [
    ('json', lambda: sample.write.mode('overwrite').json(str(write_out / 'json'))),
    ('csv', lambda: sample.write.mode('overwrite').option('header', True).csv(str(write_out / 'csv'))),
    ('parquet', lambda: sample.write.mode('overwrite').parquet(str(write_out / 'parquet'))),
    ('iceberg', lambda: sample.writeTo('local.default.taxi_write_bench').using('iceberg').createOrReplace()),
]:
    spark.catalog.clearCache()
    import time
    start = time.perf_counter()
    writer()
    write_results[fmt] = time.perf_counter() - start

for fmt, sec in sorted(write_results.items(), key=lambda x: x[1]):
    print(f'{fmt:10s} write: {sec:.2f}s')

## Key Takeaways

1. Columnar formats (Parquet, Iceberg) outperform row-based (JSON, CSV).
2. Vectorized Parquet reader is significantly faster than serial.
3. Iceberg adds metadata overhead on write but enables pruning and ACID on read.